# DSAR × Spark Declarative Pipelines · 00 · Setup & landing source

**What this sub-solution is.** Allegiant's Merlot ingest runs as a Spark
Declarative Pipeline (SDP):

```
autoloader → obfuscation view → bronze STREAMING TABLE → silver (AUTO CDC / SCD1) → gold
```

Bronze is an **append-only streaming source**. When a per-subject DSAR erasure
runs an in-place `UPDATE`/`DELETE` on bronze, that is a **non-append commit** —
the downstream streaming flow fails fatally with:

> *Streaming tables may only use append-only streaming sources... we detected an
> update or delete to one or more rows in the source table.*

**This solution hardens the pipeline so that never happens**, and wires the
existing per-subject erasure engine in so a DSAR request erases **every layer**
(bronze + silver + gold) + physically purges, **with zero pipeline downtime and
no full refresh**.

This notebook (`00`) is a **batch** notebook — run it once to build an isolated
demo schema and the **raw landing table** that the SDP pipeline reads. It does
not define the pipeline itself; `01_sdp_pipeline` does that.

> Self-contained & idempotent: every run drops and rebuilds the demo schema.
> All names are widget-driven — nothing here is Allegiant-specific.

## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (isolated SDP demo)")
dbutils.widgets.text("num_users", "2000", "3 Number of customers")
dbutils.widgets.text("events_per_user", "5", "4 Raw events per customer")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
N_USERS = int(dbutils.widgets.get("num_users"))
N_EV    = int(dbutils.widgets.get("events_per_user"))

print("Target schema :", FQ)
print("Customers     :", N_USERS)
print("Raw events    :", N_USERS * N_EV)

## 1. Clean rebuild of the isolated demo schema

Idempotent: drop + recreate so every run is a clean baseline. The SDP pipeline
writes its own bronze/silver/gold tables **into this same schema** at runtime.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"DROP SCHEMA IF EXISTS {FQ} CASCADE")
spark.sql(f"CREATE SCHEMA {FQ} COMMENT 'Isolated demo: DSAR erasure integrated with a Spark Declarative Pipeline'")
print("Recreated", FQ)

## 2. The raw landing table (what autoloader would write)

`raw_user` stands in for the files an autoloader lands. It carries:

- `user_id` — the **stable business key**. This is the authoritative join key the
  whole pipeline keys on, and it is **never PII / never obfuscated** — so it
  still resolves a subject *after* PII has been masked. (Email is the DSAR intake
  key; we resolve email → `user_id` once, then erase by `user_id` everywhere.)
- `email`, `full_name` — **PII**, obfuscated by the view before bronze.
- `profile_json` — a nested blob with `contact.email` + `contact.name` PII
  (the in-JSON masking case Allegiant cares about).
- `revenue`, `event_ts`, `_ingest_ts` — non-PII payload that must be **preserved**.

It is a plain Delta table appended to over time — exactly an append-only source.

In [ ]:
from pyspark.sql import functions as F

base = (
    spark.range(N_USERS)
    .withColumn("user_id", F.concat(F.lit("U"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("first", F.element_at(F.array(
        *[F.lit(x) for x in ["Alex","Sam","Jordan","Taylor","Morgan","Casey","Riley","Jamie","Drew","Quinn"]]),
        (F.col("id") % 10 + 1).cast("int")))
    .withColumn("last", F.element_at(F.array(
        *[F.lit(x) for x in ["Lucero","Ortiz","Nguyen","Patel","Kim","Diaz","Reed","Cole","Shah","Vega"]]),
        (F.col("id") % 10 + 1).cast("int")))
    .withColumn("full_name", F.concat_ws(" ", "first", "last"))
    .withColumn("email", F.concat_ws("", F.lower(F.col("first")), F.lit("."),
                                     F.lower(F.col("last")), F.col("id").cast("string"),
                                     F.lit("@example.com")))
)

# explode into N_EV raw events per user
events = (
    base.withColumn("evt", F.explode(F.sequence(F.lit(0), F.lit(N_EV - 1))))
    .withColumn("event_id", F.concat_ws("-", "user_id", F.col("evt").cast("string")))
    .withColumn("revenue", F.round(F.rand(7) * 500, 2))
    .withColumn("event_ts", F.expr("current_timestamp() - make_interval(0,0,0,cast(evt as int),0,0,0)"))
    # per-event ingest time (evt increases => later ingest) so SCD1 sequence_by is
    # deterministic: the highest-evt row per user survives in silver.
    .withColumn("_ingest_ts", F.expr("current_timestamp() + make_interval(0,0,0,0,0,cast(evt as int),0)"))
    .withColumn("profile_json", F.to_json(F.struct(
        F.struct(F.col("email").alias("email"), F.col("full_name").alias("name")).alias("contact"),
        F.struct(F.lit("gold").alias("tier"), F.col("revenue").alias("ltv")).alias("loyalty"))))
    .select("event_id", "user_id", "email", "full_name", "profile_json",
            "revenue", "event_ts", "_ingest_ts")
)

(events.write.mode("overwrite").saveAsTable(f"{FQ}.raw_user"))
print("raw_user rows:", spark.table(f"{FQ}.raw_user").count())
display(spark.table(f"{FQ}.raw_user").limit(5))

## 3. Pick a demo DSAR subject

We surface one real subject (email + resolved `user_id`) so the erasure notebook
(`02`) has a concrete target. In production the email arrives from OneTrust /
the `dsar_request` table (see the base `dsar_erasure/` layer).

In [ ]:
subject = spark.table(f"{FQ}.raw_user").orderBy("user_id").limit(1).collect()[0]
print("Demo DSAR subject")
print("  email  :", subject["email"])
print("  user_id:", subject["user_id"], "(stable key — survives obfuscation)")
print("  name   :", subject["full_name"])
print()
print("Use this email as the 'subject_email' widget in 02_zero_downtime_erasure.")

## 4. Summary & next step

`raw_user` is ready. Next:

1. **`01_sdp_pipeline`** — attach as the source of a Declarative Pipeline; it
   builds `bronze_user` → `silver_user` → `gold_user`, hardened with
   `skipChangeCommits` on every streaming hop.
2. **`02_zero_downtime_erasure`** — run a DSAR erasure across all layers while
   the pipeline keeps running.